In [ ]:
# ============================================================ #
# Cell 0: Download resume state từ Google Drive (chỉ chạy khi resume)
# ============================================================ #
# Đặt TRAIN_PHASE ở đây:
#   1  -> train task 0-1, lưu continuation_state_task_1.pt
#   2  -> load task_1 state, train task 2, lưu continuation_state_task_2.pt
#   3  -> load task_2 state, train task 3, lưu continuation_state_task_3.pt
#   4  -> load task_3 state, train task 4-5 (xong)
#   5  -> train toàn bộ task 0-5 (không resume)
# ============================================================ #
import os
import subprocess
import sys
import zipfile

TRAIN_PHASE = 5  # <<< SỬA Ở ĐÂY: 1=task0-1, 2=task2, 3=task3, 4=task4-5, 5=all

PHASE_CONFIG = {
    1: {
        "task_start": 0,
        "task_end": 1,
        "save_resume_after_task": 1,
        "resume_file": None,
    },
    2: {
        "task_start": 2,
        "task_end": 2,
        "save_resume_after_task": 2,
        "resume_file": "continuation_state_task_1.pt",
    },
    3: {
        "task_start": 3,
        "task_end": 3,
        "save_resume_after_task": 3,
        "resume_file": "continuation_state_task_2.pt",
    },
    4: {
        "task_start": 4,
        "task_end": 5,
        "save_resume_after_task": None,
        "resume_file": "continuation_state_task_3.pt",
    },
    5: {
        "task_start": 0,
        "task_end": 5,
        "save_resume_after_task": None,
        "resume_file": None,
    },
}

phase_config = PHASE_CONFIG[TRAIN_PHASE]
desired_resume_file = phase_config["resume_file"]
target = None

if desired_resume_file is None:
    print(f"Phase {TRAIN_PHASE}: không cần resume state.")
else:
    os.makedirs("/tmp/next/continue", exist_ok=True)

    # <<< ĐỔI URL NÀY thành link Google Drive chứa file .zip resume state >>>
    GDRIVE_URL = "https://drive.google.com/file/d/YOUR_FILE_ID/view?usp=sharing"

    archive_path = "/tmp/next/continue/continue.zip"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    subprocess.run(
        [sys.executable, "-m", "gdown", "--fuzzy", GDRIVE_URL, "-O", archive_path],
        check=True,
    )

    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall("/tmp/next/")

    resume_candidates = []
    for root, _, files in os.walk("/tmp/next"):
        for f in files:
            if f.endswith(".pt"):
                resume_candidates.append(os.path.join(root, f))

    print("PT files found:")
    for p in resume_candidates:
        print(p)

    for p in resume_candidates:
        if os.path.basename(p).lower() == desired_resume_file:
            target = p
            break

    if target is None:
        raise FileNotFoundError(
            f"Phase {TRAIN_PHASE} yêu cầu {desired_resume_file} nhưng không tìm thấy trong /tmp/next."
        )

print("Selected resume path:", target)
if target is not None:
    print("exists:", os.path.exists(target))
    print("size:", os.path.getsize(target))

In [ ]:
"""
Federated Class Incremental Learning - Training Entry Point (Kaggle)
=====================================================================
Chỉnh CONFIG bên dưới rồi Run All. Cell 0 đã set TRAIN_PHASE và `target`.
"""

import os
import sys

# =============================================================================
# KAGGLE SETUP - Clone from GitHub (luôn fresh để lấy code mới nhất)
# =============================================================================
REPO_PATH = "/tmp/FL_IL_IDS"


def setup_imports():
    import shutil
    if os.path.exists(REPO_PATH):
        print(f"Removing stale clone at {REPO_PATH}...")
        shutil.rmtree(REPO_PATH)
    print("Cloning from GitHub...")
    os.system(f"git clone https://github.com/khoilv2005/FL_IL_IDS.git {REPO_PATH}")
    new_sys_path = [p for p in sys.path if not p.startswith("/kaggle/input")]
    sys.path = [REPO_PATH] + new_sys_path
    for k in [k for k in sys.modules if "fed_learning" in k]:
        del sys.modules[k]
    print(f"sys.path[0]: {sys.path[0]}")


setup_imports()


# =============================================================================
# CONFIGURATION
# =============================================================================
CONFIG = {
    # ------------------------------------------------------------------
    # Data
    # ------------------------------------------------------------------
    "data_dir": "/kaggle/input/datasets/khoilv2005/100-clients/100-clients",
    "random_seed": 42,

    # ------------------------------------------------------------------
    # Training mode + algorithm
    # mode:      "fed_il" | "il" | "decentralized"
    # algorithm: fed_il -> "cgofed" | "fedavg_ewc" | "fedprox_ewc" | "fedavg_lwf"
    #                      "fedprox_lwf" | "fedcbdr" | "der" | "nice" | "glfc"
    #                      "refed" | "dfca_il"
    #            decentralized -> "plexus" | "denice"
    #            il            -> "ewc" | "lwf" | "der" | "nice" | "denice"
    # ------------------------------------------------------------------
    "mode": "decentralized",
    "algorithm": "denice",

    # ------------------------------------------------------------------
    # Output — /kaggle/working/ tự động download được từ tab Output
    # ------------------------------------------------------------------
    "output_dir": "/kaggle/working/results_denice",

    # ------------------------------------------------------------------
    # Split-run / continuation — lấy từ Cell 0 (phase_config + target)
    # ------------------------------------------------------------------
    "task_start": phase_config["task_start"],
    "task_end":   phase_config["task_end"],
    "save_resume_after_task": phase_config["save_resume_after_task"],
    "resume_state_path": target,          # None nếu Phase 5 / không resume
    "resume_output_dir": "/kaggle/working/results_denice",

    # ------------------------------------------------------------------
    # Incremental learning setup — 6 tasks × (6,6,6,6,6,4) = 34 classes
    # ------------------------------------------------------------------
    "num_clients": 100,
    "total_classes": 34,
    "base_classes": 6,
    "classes_per_task": 6,

    # ------------------------------------------------------------------
    # Common hyper-parameters
    # ------------------------------------------------------------------
    "mu_fedprox": 0.0,
    "rounds_per_task": 20,
    "local_epochs": 1,
    "learning_rate": 0.001,
    "batch_size": 2048,
    "eval_batch_size": 8192,
    # eval_every > rounds_per_task -> bỏ eval giữa round.
    # denice_post_task_eval=False -> bỏ eval cuối task, chỉ train và lưu checkpoint.
    "eval_every": 9999,
    "round_checkpoint_every": 1,

    # ------------------------------------------------------------------
    # CGoFed
    # ------------------------------------------------------------------
    "mu_cgofed": 1.0,
    "lambda_decay": 0.8,
    "theta_threshold": 0.35,
    "cross_task_weight": 0.3,
    "lambda_cross_task": 0.3,
    "energy_threshold": 0.99,
    "num_samples_rep": 1000,
    "top_k": 2,

    # EWC
    "ewc_lambda": 1000.0,
    "fisher_samples": 200,
    "online_ewc": False,

    # LwF
    "lwf_alpha": 1.0,
    "temperature": 2.0,
    "lwf_alpha_scale": 1.0,
    "distill_old_classes_only": False,

    # FedCBDR
    "tau_old": 0.9,
    "tau_new": 1.1,
    "omega_old": 1.1,
    "omega_new": 0.9,
    "buffer_size": 500,
    "replay_ratio": 0.5,
    "rne_feature_batch_size": 1024,
    "seed": 42,

    # DER
    "lambda_aux": 1.0,
    "lambda_sparsity": 0.1,
    "s_max": 15.0,
    "der_temperature": 2.0,
    "der_stage1_rounds": 12,
    "der_stage2_rounds": 8,

    # ------------------------------------------------------------------
    # NICE / DeNICE
    # ------------------------------------------------------------------
    "tau": 0.95,
    "nice_max_phases": 20,
    "nice_phase_epochs": 1,
    "memo_per_class": 50,
    "nice_context_eval": True,
    "nice_debug_context_detector": True,

    # DeNICE adapter layers
    # ["fc1"]                 -> Phase 1 MVP (nhẹ nhất)
    # ["fc1", "gru"]          -> Phase 2a
    # ["fc1", "gru", "conv3"] -> Phase 2b (đầy đủ)
    "denice_adapter_layers": ["fc1", "gru", "conv3"],
    "denice_debug": True,
    "denice_save_round_artifacts": False,
    "denice_checkpoint_format": "delta",

    # Train-time eval guard. Full all-client eval should be run offline after training.
    "denice_post_task_eval": False,
    "denice_eval_max_clients": None,
    "denice_eval_require_full_coverage": True,
    "denice_eval_max_samples": None,
    "denice_eval_progress_every_clients": 10,  # in progress mỗi 10 clients
    "denice_eval_progress_every_batches": 0,

    # DeNICE context routing bank. scope="cluster" follows the proposal:
    # share context capsule/sketches only inside decentralized collaboration group.
    # Use scope="global" only for ablation/debug.
    "denice_shared_context_eval": True,
    "denice_shared_context_scope": "cluster",
    "denice_shared_context_max_per_episode": 512,

    # DeNICE decentralized aggregation (Đề xuất §6-§7). Mặc định giữ hành vi cũ.
    # denice_aggregation_method: "weighted_mean" (mặc định, alpha_ij chuẩn)
    #                            | "coordinate_median" | "trimmed_mean" (robust §7)
    "denice_aggregation_method": "weighted_mean",
    "denice_aggregation_trim_ratio": 0.1,
    "denice_aggregation_count_transform": "log",
    "denice_aggregation_self_floor": 0.25,
    "denice_gamma": 0.15,
    # G_i = {j | cùng cluster AND s_ij > delta} (§6). True = lọc theo đồ thị context.
    "denice_collab_use_context_edges": True,
    "denice_require_label_overlap": True,
    "denice_centroid_gate_threshold": 0.75,
    "denice_cluster_delta_sim": 0.0,  # <=0 dùng adaptive threshold
    "denice_cluster_edge_top_k": 20,
    "denice_cluster_edge_quantile": 0.40,
    "denice_cluster_min_signal_std": 0.02,
    "denice_cluster_theta_s": 0.5,

    # DeNICE graceful recycling
    "denice_enable_recycling": True,
    "denice_recycle_ratio": 0.02,
    "denice_recycle_min": 1,
    "denice_recycle_max_per_layer": 8,
    "denice_recycle_grace_tasks": 1,
    "denice_recycle_usage_recent_threshold": 0.10,
    "denice_recycle_max_old_metric_drop": 0.02,
    "denice_recycle_require_old_check": True,

    # ------------------------------------------------------------------
    # GLFC
    # ------------------------------------------------------------------
    "glfc_memory_size": 2000,
    "glfc_entropy_threshold": 1.2,
    "glfc_distill_weight": 0.5,
    "glfc_recon_iters": 250,
    "glfc_num_recon_images": 20,

    # Re-Fed
    "refed_memory_size": 2000,
    "refed_lambda_pim": 0.5,
    "refed_pim_iterations": 5,

    # ------------------------------------------------------------------
    # Plexus
    # ------------------------------------------------------------------
    "plexus_sample_size": 10,
    "plexus_num_aggregators": 1,
    "plexus_success_fraction": 0.8,
    "plexus_inactivity_threshold": 50,
    "plexus_scale_clients": True,
    "plexus_initial_client_ratio": 0.5,
    "plexus_final_client_ratio": 1.0,
}


# =============================================================================
# MAIN — chạy trực tiếp (không cần if __name__ == "__main__" trong Jupyter)
# =============================================================================
from fed_learning.training.task_loop import run_incremental_training

run_incremental_training(CONFIG)